# Plot the QuantUI results from the GPU session

In the GPU session you produced two datasets and wrote them to
`$COURSE_WORK/quantui`:

1. **CPU vs GPU wall times** (`run_cpu_gpu_comparison.py`) &rarr; a bar chart, and
2. **a geometry-relaxation trajectory** (`run_geometry_optimization.py`) &rarr; a line plot.

This notebook is deliberately thin: every plotting decision lives in
[`quantui_result_plots.py`](quantui_result_plots.py) next to it, so the logic is
readable and version-controlled rather than buried in notebook cells. Here we
only import and call.

If your own results aren't present (a queued job, a short break), it falls back
to bundled sample data so the plotting lesson still runs — the figure title says
which one you're looking at.

In [ ]:
%matplotlib inline
import sys, pathlib
# Make sure the sidecar module beside this notebook is importable.
sys.path.insert(0, str(pathlib.Path.cwd()))

from quantui_result_plots import (
    find_result_source,
    load_comparison_results,
    plot_compute_time_bars,
    load_trajectory,
    plot_relaxation_trajectory,
    save_figure,
)

source, is_sample = find_result_source()
print("Reading QuantUI results from:", source)
print("  -> bundled SAMPLE data" if is_sample else "  -> your own GPU-session results")

## 1. CPU vs GPU wall time — where does the GPU start to win?

Grouped bars, one pair per system. Watch the GPU bars stay nearly flat while the
CPU bar for the largest basis set towers: that flat stretch is fixed launch and
transfer overhead, and the crossover is the system where the two bars are level.

In [ ]:
results = load_comparison_results(source)
fig, ax = plot_compute_time_bars(results, is_sample=is_sample)
save_figure(fig, "cpu_gpu_walltime")

## 2. Geometry relaxation — energy vs optimization step

A geometry optimization moves the atoms downhill in energy until the forces
vanish. The line falls steeply at first, then levels off as the structure
reaches its minimum. (This is a CPU calculation — it does not need a GPU.)

In [ ]:
traj = load_trajectory(source, preset="water")
if traj is None:
    print("No geometry_optimization_water.json found — run run_geometry_optimization.py first.")
else:
    fig, ax = plot_relaxation_trajectory(traj)
    save_figure(fig, "geometry_relaxation")

Both figures were saved next to this notebook as **PNG** (for slides) and
**PDF** (vector, for print). Before using one in a report, run it past the
figure checklist in the [session README](README.md): does it answer one
question, are the axes and units labelled, and — for the timings — is the CPU
allocation named, since a speedup without its denominator is not a result?